In [ ]:
import random
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

random.seed(42)
np.random.seed(42)

CATEGORIES = ["grocery", "food_delivery", "recharge", "bill_payment", "travel",
              "ecommerce", "entertainment"]
REGIONS = ["North", "South", "East", "West"]
METHODS = ["UPI", "Wallet", "Card", "Netbanking"]
METHOD_WEIGHTS = [0.55, 0.20, 0.15, 0.10]
AMOUNTS_INR = [49, 99, 149, 299, 499, 799, 1499, 2999, 4999]
AMOUNT_WEIGHTS = [0.18, 0.16, 0.14, 0.14, 0.12, 0.10, 0.08, 0.05, 0.03]

# --- 40 merchants ---
merchants = pd.DataFrame({
    "merchant_id": range(1, 41),
    "merchant_name": [f"Merchant_{i:03d}" for i in range(1, 41)],
    "category": [random.choice(CATEGORIES) for _ in range(40)],
    "region": [random.choice(REGIONS) for _ in range(40)],
})

# --- 350 established users, signed up 30-730 days before the window start ---
window_start = datetime(2026, 1, 1)
users = pd.DataFrame({
    "user_id": range(1, 351),
    "signup_date": [window_start - timedelta(days=random.randint(30, 730)) for _ in range(350)],
})

# --- 500 baseline transactions over a 30-day window ---
rows = []
for i in range(500):
    txn_time = window_start + timedelta(
        days=random.randint(0, 29), hours=random.randint(0, 23), minutes=random.randint(0, 59))
    status = np.random.choice(["captured", "failed", "chargeback"], p=[0.92, 0.06, 0.02])
    rows.append({
        "transaction_id": f"TXN{100000+i}",
        "user_id": random.randint(1, 350),
        "merchant_id": random.randint(1, 40),
        "transaction_time": txn_time,
        "amount_inr": np.random.choice(AMOUNTS_INR, p=AMOUNT_WEIGHTS),
        "payment_method": np.random.choice(METHODS, p=METHOD_WEIGHTS),
        "status": status,
        "risk_score": random.randint(0, 100),
    })

# --- inject 15 "burner account" chargeback frauds: brand-new users (< 30 days old) ---
next_user_id = 351
for i in range(15):
    txn_time = window_start + timedelta(days=random.randint(10, 29), hours=random.randint(0, 23))
    signup = txn_time - timedelta(days=random.randint(1, 25))
    users = pd.concat([users, pd.DataFrame([{"user_id": next_user_id, "signup_date": signup}])],
                       ignore_index=True)
    rows.append({
        "transaction_id": f"TXN{200000+i}",
        "user_id": next_user_id,
        "merchant_id": random.randint(1, 40),
        "transaction_time": txn_time,
        "amount_inr": random.choice([999, 1999, 2999, 4999]),
        "payment_method": "Card",
        "status": "chargeback",
        "risk_score": random.randint(70, 100),
    })
    next_user_id += 1

# --- inject 8 velocity-attack clusters: 4 rapid-fire txns each within a 5-minute window ---
for cluster in range(8):
    victim_user = random.randint(1, 350)
    base_time = window_start + timedelta(days=random.randint(0, 29), hours=random.randint(0, 23))
    for k in range(4):
        rows.append({
            "transaction_id": f"TXN{300000 + cluster*4 + k}",
            "user_id": victim_user,
            "merchant_id": random.randint(1, 40),
            "transaction_time": base_time + timedelta(minutes=k),
            "amount_inr": random.choice([299, 399, 499]),
            "payment_method": "Card",
            "status": "captured" if k == 3 else "failed",
            "risk_score": random.randint(60, 95),
        })

ledger = pd.DataFrame(rows)  # 500 + 15 + 32 = 547 rows
merchants.to_csv("merchants.csv", index=False)
users.to_csv("users.csv", index=False)
ledger.to_csv("ledger.csv", index=False)

# --- build the deliberately-discrepant "gateway export" copy for reconciliation ---
gateway = ledger.copy()
n = len(gateway)
missing_idx = np.random.choice(n, size=int(0.05 * n), replace=False)
gateway = gateway.drop(index=missing_idx).reset_index(drop=True)

mismatch_idx = np.random.choice(len(gateway), size=int(0.03 * n), replace=False)
gateway.loc[mismatch_idx, "amount_inr"] = gateway.loc[mismatch_idx, "amount_inr"] + \
    np.random.choice([-100, -50, 50, 100], size=len(mismatch_idx))

extra_rows = []
for i in range(int(0.02 * n)):
    extra_rows.append({
        "transaction_id": f"TXNX{9000+i}", "user_id": random.randint(1, 350),
        "merchant_id": random.randint(1, 40),
        "transaction_time": window_start + timedelta(days=random.randint(0, 29)),
        "amount_inr": random.choice(AMOUNTS_INR), "payment_method": random.choice(METHODS),
        "status": "captured", "risk_score": random.randint(0, 100),
    })
gateway = pd.concat([gateway, pd.DataFrame(extra_rows)], ignore_index=True)

status_idx = np.random.choice(len(gateway), size=int(0.02 * n), replace=False)
gateway.loc[status_idx, "status"] = "failed"

gateway.to_csv("gateway_export.csv", index=False)


### Exporting DataFrames to Excel Files

The following code will save the `merchants`, `users`, `ledger`, and `gateway` DataFrames to separate Excel files (.xlsx). CSV files for these DataFrames were already created in the previous step.

# New Section

In [ ]:
# Export DataFrames to Excel files
merchants.to_excel("merchants.xlsx", index=False)
users.to_excel("users.xlsx", index=False)
ledger.to_excel("ledger.xlsx", index=False)
gateway.to_excel("gateway_export.xlsx", index=False)

print("DataFrames successfully exported to Excel files:")
print("- merchants.xlsx")
print("- users.xlsx")
print("- ledger.xlsx")
print("- gateway_export.xlsx")

print("\nAnd CSV files (created in previous step):")
print("- merchants.csv")
print("- users.csv")
print("- ledger.csv")
print("- gateway_export.csv")

DataFrames successfully exported to Excel files:
- merchants.xlsx
- users.xlsx
- ledger.xlsx
- gateway_export.xlsx

And CSV files (created in previous step):
- merchants.csv
- users.csv
- ledger.csv
- gateway_export.csv


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import files

print("Please upload 'ledger.csv' and 'gateway_export.csv'")
uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')


Please upload 'ledger.csv' and 'gateway_export.csv'


After uploading the files, please re-run the cell with the reconciliation code (the one that generated the `FileNotFoundError`).

### Payment Reconciliation Function

This section defines a reusable function `reconcile_payments` to identify discrepancies between a `ledger_df` and a `gateway_df`. It will look for missing transactions, extra transactions, and mismatches in `amount_inr` and `status`.

In [1]:
def reconcile_payments(ledger_df, gateway_df):
    # Ensure transaction_id is the key for set operations
    ledger_txns = set(ledger_df['transaction_id'])
    gateway_txns = set(gateway_df['transaction_id'])

    # 1. Transactions missing in gateway export
    missing_in_gateway_ids = list(ledger_txns - gateway_txns)
    missing_in_gateway = ledger_df[ledger_df['transaction_id'].isin(missing_in_gateway_ids)].copy()

    # 2. Transactions missing in ledger (extra in gateway)
    extra_in_gateway_ids = list(gateway_txns - ledger_txns)
    extra_in_gateway = gateway_df[gateway_df['transaction_id'].isin(extra_in_gateway_ids)].copy()

    # For mismatches, consider only transactions present in both
    common_txns = list(ledger_txns.intersection(gateway_txns))

    # Merge common transactions to compare details
    merged_df = pd.merge(ledger_df[ledger_df['transaction_id'].isin(common_txns)],
                         gateway_df[gateway_df['transaction_id'].isin(common_txns)],
                         on='transaction_id',
                         suffixes=('_ledger', '_gateway'))

    # 3. Amount mismatches
    amount_mismatches = merged_df[merged_df['amount_inr_ledger'] != merged_df['amount_inr_gateway']].copy()
    if not amount_mismatches.empty:
        amount_mismatches['amount_difference'] = amount_mismatches['amount_inr_ledger'] - amount_mismatches['amount_inr_gateway']
        amount_mismatches = amount_mismatches[['transaction_id', 'amount_inr_ledger', 'amount_inr_gateway', 'amount_difference']]

    # 4. Status mismatches
    status_mismatches = merged_df[merged_df['status_ledger'] != merged_df['status_gateway']].copy()
    if not status_mismatches.empty:
        status_mismatches = status_mismatches[['transaction_id', 'status_ledger', 'status_gateway']]

    return missing_in_gateway, extra_in_gateway, amount_mismatches, status_mismatches
